# 5.2. Implementation of Multilayer Perceptrons
D2L의 Implementation of Multilayer Perceptrons장을 PyTorch 기준으로 정리함.


## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. MLP 구현하기

앞 장들에서는 MLP 구조랑 활성화 함수가 왜 필요한지 배웠다.

이번에는 Fashion-MNIST 데이터에 MLP를 실제로 구현해보겠다.

전체 구조는 이렇다.

```text
입력 이미지
[batch_size, 1, 28, 28]

        ↓ Flatten

[batch_size, 784]

        ↓ Linear

[batch_size, 256]

        ↓ ReLU

[batch_size, 256]

        ↓ Linear

[batch_size, 10]
```

이번 모델은 전과 같다.

Softmax Regression과 차이는 입력층과 출력층 사이에 은닉층과 활성화 함수가 추가됬다는 것이다.

## 2. 데이터 준비

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

## 3. MLP 직접 구현

D2L에서는 먼저 W1, b1, W2, b2를 직접 만들고 ReLU까지 직접구현한다.

은닉층이 하나인 MLP에는 두 번의 선형 변환이 필요하다.

첫번째 선형변환:
```text
입력 784개 -> 은닉 neuron 256개
```

두번째 선형변환:
```text
은닉 neuron 256개 -> 출력 class 10개
```

따라서 두 개의 weight와 두 개의 bias가 필요하다.

W1 : [784, 256]
b1 : [256]

W2 : [256, 10]
b2 : [10]

계산은 이렇게 진행된다.

    X => X @ W1 + b1 => ReLU => H @ W2 + b2 => logits

In [ ]:
class MLPScratch(nn.Module):
    def __init__(self):
        super().__init__()

        self.W1 = nn.Parameter(
            torch.randn(784, 256) * 0.01
        )
        self.b1 = nn.Parameter(
            torch.zeros(256)
        )

        self.W2 = nn.Parameter(
            torch.randn(256, 10) * 0.01
        )
        self.b2 = nn.Parameter(
            torch.zeros(10)
        )

    def relu(self, X):
        return torch.maximum(
            X,
            torch.zeros_like(X)
        )

    def forward(self, X):

        X = X.reshape(X.shape[0], -1)

        H = X @ self.W1 + self.b1
        H = self.relu(H)

        output = H @ self.W2 + self.b2

        return output

## 4. Forward 과정 이해

입력 batch가 256개라고 했을때 처음 입력은

    [256, 1, 28, 28]
이다.

펼치면
```py
X = X.reshape(X.shape[0], -1) # [256, 784] 가 된다.
```

첫번째 선형 변환:

H = X @ W1 + b1

X : [256, 784]
W1: [784, 256]

X @ W1 => H : [256, 256] 이 된다.

그다음에 ReLU를 적용한다. (shape 변경 X)

마지막 출력층에서는

output = H @ W2 + b2

H : [256, 256]
W2: [256, 10]

H @ W2 => output : [256, 10] 이 된다.

각 데이터마다 10개 클래스에 대한 점수가 출력된다. 이 최종 점수가 logit이다.

## 5. MLP 학습하기

MLP라고 학습 과정 자체가 새로 바뀌는 것은 아니다.
D2L에서도 MLP의 training loop는 Softmax Regression과 동일하다고 설명한다.

학습 과정은 이렇다.

```text
1. 데이터를 모델에 입력한다.
2. 예측값을 계산한다.
3. loss를 계산한다.
4. gradient를 초기화한다.
5. backward로 gradient를 계산한다.
6. optimizer가 parameter를 업데이트한다.
```

모델 내부가 Linear든 Linear - ReLU- Linear 든 학습 흐름은 똑같다.

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MLPScratch().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

epochs = 10

In [5]:
for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        # 1. Forward
        y_hat = model(X)

        # 2. Loss
        loss = loss_fn(y_hat, y)

        # 3. 기존 gradient 제거
        optimizer.zero_grad()

        # 4. Backward
        loss.backward()

        # 5. parameter 업데이트
        optimizer.step()

        total_loss += loss.item()

        predictions = y_hat.argmax(dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

    accuracy = correct / total

    print(
        f"Epoch {epoch + 1:2d} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Accuracy: {accuracy:.4f}"
    )

Epoch  1 | Loss: 1.0394 | Accuracy: 0.6368
Epoch  2 | Loss: 0.5986 | Accuracy: 0.7894
Epoch  3 | Loss: 0.5228 | Accuracy: 0.8165
Epoch  4 | Loss: 0.4811 | Accuracy: 0.8310
Epoch  5 | Loss: 0.4580 | Accuracy: 0.8395
Epoch  6 | Loss: 0.4342 | Accuracy: 0.8469
Epoch  7 | Loss: 0.4184 | Accuracy: 0.8532
Epoch  8 | Loss: 0.4052 | Accuracy: 0.8566
Epoch  9 | Loss: 0.3932 | Accuracy: 0.8611
Epoch 10 | Loss: 0.3833 | Accuracy: 0.8633


학습은 전과 같다.

    예측 => loss 계산 => gradient 계산 => weight 수정

## 6. PyTorch로 간결하게 구현하기

실제로는 위처럼 W1, W2를 직접 관리할 필요가 없다.  
D2L의 두 번째 구현도 `Flatten => Linear => ReLU => Linear`를 `Sequential`로 연결한다.

MLP의 내부 동작을 이해하기 위해 weight, bias를 직접 만들었는데 PyTorch에서는 `nn.Linear`를 사용하면 weight와 bias를 자동으로 관리할 수 있다고 한다.

In [ ]:
model = nn.Sequential(
    nn.Flatten(),       # 펼치기
    nn.Linear(784, 256),# 256개의 hidden feature 만들기 (내부적으로 X @ W1 + b1)
    nn.ReLU(),          # 음수 0으로 만들고 양수 그대로 두기
    nn.Linear(256, 10)  # 256개 hidden feature로 10개 class score만들기
).to(device)

In [7]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

## 7. 오늘의 정리

- MLP는 여러 개의 layer를 연결한 신경망이다.
- Fashion-MNIST 이미지는 28 × 28이므로 펼치면 784개의 feature가 된다.
- 이번 MLP는 `784 → 256 → 10` 구조를 사용했다.
- `W1`은 784개의 입력을 256개의 hidden feature로 변환한다.
- ReLU는 비선형성을 추가하며 tensor의 shape은 변경하지 않는다.
- `W2`는 256개의 hidden feature를 10개의 class score로 변환한다.
- 최종 출력 `[batch_size, 10]`의 값들을 logit이라고 한다.
- `loss.backward()`는 각 parameter의 gradient를 계산한다.
- `optimizer.step()`은 계산된 gradient를 이용해 weight와 bias를 업데이트한다.
- MLP가 되어도 전체 학습 과정은 Softmax Regression과 동일하다.
- 직접 구현한 `X @ W + b`는 PyTorch의 `nn.Linear`과 같은 역할을 한다.
- 실제 구현에서는 `nn.Sequential`을 이용해 MLP를 간단하게 구성할 수 있다.